| Step | Purpose |
|---|---|
| 7.1 | Sample 20 rows per residence `kind` × `precision`, and 20 per reason code |
| 7.2 | Label each by reading its snippet (and the page, if needed): correct / wrong place / wrong kind / unclear |
| 7.3 | Precision = correct ÷ labelled; coverage = files with ≥1 row ÷ files with a declaration |
| 7.4 | Markdown: which rules to keep, fix or drop; numbers go to DATA.md |

In [1]:
import sys
from pathlib import Path
PIPELINE_DIR = Path.cwd().parent
if str(PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPELINE_DIR))
import config
from rx import io, text, places, remote

In [ ]:
res = io.read_stage("interim", "residences")
rea = io.read_stage("interim", "reasons")
# shuffle once, then keep the first 20 rows of each group (keeps every column; pandas 3 safe)
s1 = (res[res.method == "declaration_regex"].sample(frac=1, random_state=11)
      .groupby(["kind", "precision"]).head(20))
s1 = s1.assign(table="residences", kind_or_code=s1.kind, place_or_snippet=s1.place_id + " | " + s1.snippet)
s2 = rea.sample(frac=1, random_state=11).groupby("reason_code").head(20)s2 = s2.assign(table="reasons", kind_or_code=s2.reason_code, place_or_snippet=s2.snippet.str[:250])
v = pd.concat([s1, s2])[["table", "NAID", "pageNumber", "kind_or_code", "place_or_snippet"]].assign(label="", note="")
v.to_csv(config.INTERIM_DIR / "validation.csv", index=False)
# … label in a spreadsheet, then:
v = pd.read_csv(config.INTERIM_DIR / "validation.csv")
pd.crosstab(v.kind_or_code, v.label, normalize="index").round(2)

AttributeError: 'DataFrame' object has no attribute 'kind'